# 3. Retrieval & Generation - RAG Pipeline with GPT-4o-mini

This notebook implements the core RAG (Retrieval-Augmented Generation) pipeline for medical question answering. It loads the ChromaDB vector index built in Notebook 2, retrieves relevant document chunks for each query, reranks them using a cross-encoder model, and generates answers using GPT-4o-mini. Results are saved for evaluation in Notebook 4.

In [1]:
import json
import os
import re
import time
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

import pandas as pd
from dotenv import load_dotenv

import chromadb
from llama_index.core import VectorStoreIndex, Settings
from llama_index.core.postprocessor import SentenceTransformerRerank
from llama_index.core.llms import ChatMessage
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.llms.openai import OpenAI
from llama_index.vector_stores.chroma import ChromaVectorStore

In [2]:
load_dotenv(Path("..") / ".env")

BASE_DIR = Path("..")
DATA_DIR = BASE_DIR / "data"
CHROMA_DIR = BASE_DIR / "chroma_db"
RESULTS_DIR = BASE_DIR / "results"
RESULTS_DIR.mkdir(exist_ok=True)

## 3.1 Load index and configure models

In [ ]:
# Embedding model (same as indexing)
embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-base-en-v1.5")
Settings.embed_model = embed_model

# LLM for self-consistency: temperature=0.7 for diverse votes
llm = OpenAI(model="gpt-4o-mini", max_tokens=100, temperature=0.7)
Settings.llm = llm

print("Models configured (temperature=0.7 for self-consistency)")

In [4]:
chroma_client = chromadb.PersistentClient(path=str(CHROMA_DIR))
chroma_collection = chroma_client.get_or_create_collection("pubmed_rag")
vector_store = ChromaVectorStore(chroma_collection=chroma_collection)

index = VectorStoreIndex.from_vector_store(vector_store)
print(f"Loaded index with {chroma_collection.count()} chunks")

Loaded index with 43806 chunks


## 3.2 Configure retriever and reranker

In [5]:
# Retriever: get top 20 by embedding similarity
retriever = index.as_retriever(similarity_top_k=20)

# Reranker: cross-encoder to rerank top 20 -> top 5
reranker = SentenceTransformerRerank(
    model="cross-encoder/ms-marco-MiniLM-L-6-v2",
    top_n=5
)

print("Retriever (top_k=20) + Reranker (top_n=5) configured")

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Retriever (top_k=20) + Reranker (top_n=5) configured


In [6]:
test_query = "Is anorectal endosonography valuable in dyschesia?"

# Retrieve
nodes_before_rerank = retriever.retrieve(test_query)
print(f"Before rerank: {len(nodes_before_rerank)} chunks")

# Rerank
nodes_after_rerank = reranker.postprocess_nodes(nodes_before_rerank, query_str=test_query)
print(f"After rerank: {len(nodes_after_rerank)} chunks")

for i, node in enumerate(nodes_after_rerank):
    print(f"  {i+1}. [{node.metadata.get('source_id', 'N/A')}] score={node.score:.4f}: {node.text[:100]}...")

Before rerank: 10 chunks
After rerank: 5 chunks
  1. [Document_47183] score=5.1019: Dyschesia can be provoked by inappropriate defecation movements. The aim of this prospective study w...
  2. [Document_216720] score=-0.9560: title: Dyssynergia
content: doctors. An abnormal or prolonged time of expulsion of the balloon is se...
  3. [Document_627762] score=-1.2793: title: Dyssynergia
content: a decrease in intrarectal pressure defecation can occur. Anal sphincter ...
  4. [Document_754472] score=-2.5774: title: Anismus
content: dyssynergic defecation is given as "inappropriate contraction of the pelvic ...
  5. [Document_453749] score=-3.3392: title: Medical ultrasound
content: capabilities in this area. The appendix can sometimes be seen whe...


## 3.3 Load queries

In [7]:
queries = []
with open(DATA_DIR / "pubmedqa.jsonl", "r") as f:
    for line in f:
        queries.append(json.loads(line))

print(f"Loaded {len(queries)} queries")
print(f"Example: {queries[0]['query']}")

Loaded 500 queries
Example: Is anorectal endosonography valuable in dyschesia?


## 3.4 Run RAG pipeline

In [ ]:
SYSTEM_PROMPT = """You are a medical expert answering yes/no/maybe questions based on PubMed abstracts.
Rules:
- Answer "yes" if the evidence supports the claim.
- Answer "no" if the evidence contradicts or does not support the claim.
- Answer "maybe" ONLY if the evidence is genuinely mixed or insufficient to decide.
- Most questions have a definitive answer. Prefer "yes" or "no" over "maybe".
- Respond with exactly one word: yes, no, or maybe."""

FEW_SHOT_EXAMPLES = """Example 1:
Question: Is increased gravitational stress beneficial for bone density?
Context: Studies show weight-bearing exercise increases bone mineral density by 2-8% in postmenopausal women...
Answer: yes

Example 2:
Question: Does smoking cessation reduce cardiovascular risk?
Context: After 1 year of cessation, coronary heart disease risk drops by 50%. After 15 years, risk equals that of a non-smoker...
Answer: yes

Example 3:
Question: Is homeopathy effective for treating asthma?
Context: A systematic review of 6 RCTs found no significant difference between homeopathic treatments and placebo in lung function or symptom scores...
Answer: no

Example 4:
Question: Can MRI replace biopsy for diagnosing prostate cancer?
Context: MRI showed sensitivity of 91% but specificity of only 37%. While useful for risk stratification, results were inconsistent across centers...
Answer: maybe

"""

NUM_VOTES = 5  # Self-consistency: number of votes per query

def extract_answer(text):
    """Extract yes/no/maybe from LLM response, handling verbose answers."""
    text = text.strip().lower()
    if text in ("yes", "no", "maybe"):
        return text
    match = re.search(r'\b(yes|no|maybe)\b', text, re.I)
    return match.group(1).lower() if match else "unknown"

def majority_vote(answers):
    """Return the majority answer from a list of answers."""
    from collections import Counter
    valid = [a for a in answers if a in ("yes", "no", "maybe")]
    if not valid:
        return "unknown"
    counts = Counter(valid)
    return counts.most_common(1)[0][0]

def build_prompt(query_text, contexts):
    """Build the prompt from query and retrieved contexts."""
    context_parts = []
    for i, ctx in enumerate(contexts):
        context_parts.append(f"[Document {i+1}]\n{ctx}")
    context = "\n\n".join(context_parts)
    return f"""{FEW_SHOT_EXAMPLES}Now answer this question:

Context:
{context}

Question: {query_text}

Answer:"""

def call_llm_with_retry(llm, prompt, max_retries=3):
    """Call LLM API with retry and exponential backoff for rate limits."""
    for attempt in range(max_retries):
        try:
            response = llm.chat([
                ChatMessage(role="system", content=SYSTEM_PROMPT),
                ChatMessage(role="user", content=prompt),
            ])
            return str(response.message.content)
        except Exception as e:
            if "429" in str(e) or "rate_limit" in str(e).lower():
                wait = 2 ** attempt  # 1s, 2s, 4s
                time.sleep(wait)
                continue
            raise
    # Final attempt without catching
    response = llm.chat([
        ChatMessage(role="system", content=SYSTEM_PROMPT),
        ChatMessage(role="user", content=prompt),
    ])
    return str(response.message.content)

print(f"Self-consistency configured: {NUM_VOTES} votes per query with retry logic")

In [ ]:
# PHASE 1: Retrieval + Reranking (local, sequential, fast)
print("Phase 1: Retrieving and reranking (local)...")

retrieval_data = []
for i, q in enumerate(queries):
    nodes_initial = retriever.retrieve(q["query"])
    nodes_reranked = reranker.postprocess_nodes(nodes_initial, query_str=q["query"])
    
    docs_before = list(dict.fromkeys(n.metadata.get("source_id", "") for n in nodes_initial))
    docs_after = list(dict.fromkeys(n.metadata.get("source_id", "") for n in nodes_reranked))
    contexts = [node.text for node in nodes_reranked]
    prompt = build_prompt(q["query"], contexts)
    
    retrieval_data.append({
        "idx": i,
        "id": q["id"],
        "query": q["query"],
        "ground_truth": q["ground_truth"],
        "golden_doc": q["golden_doc"],
        "docs_before_rerank": docs_before,
        "docs_after_rerank": docs_after,
        "contexts": contexts,
        "prompt": prompt,
    })
    
    if (i + 1) % 100 == 0:
        print(f"  Retrieved {i + 1}/{len(queries)}...")

print(f"Phase 1 complete: {len(retrieval_data)} queries retrieved\n")

In [ ]:
# PHASE 2: Self-consistency generation (parallel API calls, 5 votes per query)
NUM_WORKERS = 5  # Conservative to avoid rate limits (500 RPM limit)
print(f"Phase 2: Self-consistency with {NUM_VOTES} votes x {len(retrieval_data)} queries = {NUM_VOTES * len(retrieval_data)} API calls")
print(f"  Workers: {NUM_WORKERS}\n")

# Create tasks: each (query_idx, vote_idx) pair
all_votes = {}  # {query_idx: [vote1, vote2, ...]}
for item in retrieval_data:
    all_votes[item["idx"]] = [None] * NUM_VOTES

vote_tasks = []
for item in retrieval_data:
    for v in range(NUM_VOTES):
        vote_tasks.append((item["idx"], v, item["prompt"]))

errors = 0

def generate_vote(task):
    """Generate one vote for a query."""
    idx, vote_idx, prompt = task
    answer = call_llm_with_retry(llm, prompt)
    return idx, vote_idx, answer

with ThreadPoolExecutor(max_workers=NUM_WORKERS) as executor:
    futures = {executor.submit(generate_vote, task): task for task in vote_tasks}
    completed = 0
    
    for future in as_completed(futures):
        task = futures[future]
        try:
            idx, vote_idx, answer = future.result()
            all_votes[idx][vote_idx] = answer
        except Exception as e:
            idx, vote_idx, _ = task
            all_votes[idx][vote_idx] = f"ERROR: {str(e)}"
            errors += 1
        
        completed += 1
        if completed % 250 == 0:
            print(f"  Completed {completed}/{len(vote_tasks)} API calls...")

print(f"\nPhase 2 complete! {len(vote_tasks) - errors} succeeded, {errors} errors")

# Apply majority voting
generated_answers = []
vote_details = []
for item in retrieval_data:
    idx = item["idx"]
    votes_raw = all_votes[idx]
    votes_extracted = [extract_answer(v) for v in votes_raw]
    winner = majority_vote(votes_extracted)
    generated_answers.append(winner)
    vote_details.append(votes_extracted)

# Combine results
results = []
for i, item in enumerate(retrieval_data):
    results.append({
        "id": item["id"],
        "query": item["query"],
        "ground_truth": item["ground_truth"],
        "golden_doc": item["golden_doc"],
        "generated_answer": generated_answers[i],
        "votes": vote_details[i],
        "docs_before_rerank": item["docs_before_rerank"],
        "docs_after_rerank": item["docs_after_rerank"],
        "contexts": item["contexts"],
    })

# Quick stats
from collections import Counter
vote_unanimity = sum(1 for vd in vote_details if len(set(v for v in vd if v in ("yes","no","maybe"))) == 1)
print(f"\nTotal results: {len(results)}")
print(f"Unanimous votes: {vote_unanimity}/{len(results)} ({vote_unanimity/len(results):.1%})")

## 3.5 Save results

In [ ]:
# Save full results as JSONL
results_path = RESULTS_DIR / "rag_results.jsonl"
with open(results_path, "w") as f:
    for r in results:
        f.write(json.dumps(r) + "\n")
print(f"Saved {len(results)} results to {results_path}")

# Also save a summary CSV
summary = []
for r in results:
    predicted = r["generated_answer"]  # already extracted via majority vote
    summary.append({
        "id": r["id"],
        "query": r["query"],
        "ground_truth": r["ground_truth"],
        "predicted": predicted,
        "correct": predicted == r["ground_truth"].lower(),
        "votes": str(r["votes"]),
        "num_docs_retrieved": len(r["docs_after_rerank"]),
    })

df_summary = pd.DataFrame(summary)
df_summary.to_csv(RESULTS_DIR / "rag_summary.csv", index=False)

print(f"\nQuick accuracy check:")
print(f"  Accuracy: {df_summary['correct'].mean():.2%}")
print(f"\nPrediction distribution:")
print(df_summary['predicted'].value_counts())
print(f"\nGround truth distribution:")
print(df_summary['ground_truth'].value_counts())

In [11]:
print("=== Spot Check: 5 Random Examples ===\n")
for r in results[:5]:
    print(f"ID: {r['id']}")
    print(f"Q: {r['query']}")
    print(f"Ground Truth: {r['ground_truth']}")
    print(f"Generated: {r['generated_answer'][:300]}")
    print(f"Docs retrieved: {r['docs_after_rerank'][:3]}...")
    print("-" * 80)

=== Spot Check: 5 Random Examples ===

ID: pubmed_1
Q: Is anorectal endosonography valuable in dyschesia?
Ground Truth: yes
Generated: yes
Docs retrieved: ['Document_47183', 'Document_216720', 'Document_627762']...
--------------------------------------------------------------------------------
ID: pubmed_2
Q: Is there a connection between sublingual varices and hypertension?
Ground Truth: yes
Generated: yes
Docs retrieved: ['Document_48843', 'Document_51247', 'Document_1005901']...
--------------------------------------------------------------------------------
ID: pubmed_3
Q: Is the affinity column-mediated immunoassay method suitable as an alternative to the microparticle enzyme immunoassay method as a blood tacrolimus assay?
Ground Truth: yes
Generated: maybe
Docs retrieved: ['Document_925677', 'Document_750458', 'Document_837570']...
--------------------------------------------------------------------------------
ID: pubmed_4
Q: Does a physician's specialty influence the recording

Results saved to `results/rag_results.jsonl` (full results) and `results/rag_summary.csv` (summary with accuracy). The next notebook (04) will evaluate retrieval quality and answer quality with RAGAS.